Notebook specifically for a slide or two in the US Army ERDC CHL 2026 Intern Symposium. Goal is to show Example of the UDS interpolation error, and some dredge cut differences

In [ ]:
%reload_ext autoreload
%autoreload 2
import os
import sys
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt

# from interpolators.TIN import TIN
from interpolators.TIN2 import TIN

import numpy as np
import xdem
import geoutils as gu

import warnings
# remove future warning of spline mindist since we aren't even providing it anyways
warnings.filterwarnings('ignore', message='.*mindist parameter.*', category=FutureWarning)     

In [ ]:
WORK_DIR = Path('/mnt/teamgroup/AutoDBM/Public')
MBES = WORK_DIR / 'Multibeam_Validation_Points_For_Distribution'
SBES = WORK_DIR / 'Synthetic_Singlebeam_Surveys_For_Distribution'
CRS = pd.read_csv(WORK_DIR / 'survey_CRS_list.csv')

mbes = list((WORK_DIR / 'Multibeam_Validation_Points_For_Distribution').glob('*.csv'))
sbes = list((WORK_DIR / 'Synthetic_Singlebeam_Surveys_For_Distribution').glob('*.csv'))
crs_data = WORK_DIR / 'survey_CRS_list.csv'

geojson = Path(os.getcwd()).parent / 'shapefiles' / 'National_Channel_Framework_-6742069550782745925.geojson'
ncf = gpd.read_file(geojson)

In [ ]:
# TODO: Build code to read the crs_data and use that for CRS instead of relying on eHydro downloaded data

In [ ]:
# Build a boundary catalog from SurveyJob layers, then match each survey to the best boundary by SurveyId
data_pairs = {}

def _norm(s):
    return "".join(ch for ch in str(s).upper() if ch.isalnum())

def _tokens(s):
    return {t for t in str(s).upper().replace("-", "_").split("_") if t}

# 1) Read all boundary candidates once
boundary_catalog = []
for gdb_file in WORK_DIR.glob("*/*.gdb"):
    try:
        sj = gpd.read_file(gdb_file, layer="SurveyJob")
    except Exception:
        continue

    if "SurveyId" not in sj.columns or sj.empty:
        continue

    # keep one record per SurveyId (first occurrence)
    for sid, grp in sj.groupby("SurveyId", dropna=True):
        boundary_catalog.append({
            "survey_id": str(sid),
            "survey_norm": _norm(sid),
            "survey_tokens": _tokens(sid),
            "gdb_file": gdb_file,
            "boundary": grp.copy()
        })

# 2) Match each survey from CRS table to best boundary
for _, items in pd.read_csv(crs_data).iterrows():
    survey = items["Survey"]

    sbes_match = [p for p in sbes if survey in str(p)]
    mbes_match = [p for p in mbes if survey in str(p)]
    if not sbes_match or not mbes_match:
        print(f"Missing SBES/MBES file for: {survey}")
        continue

    s_norm = _norm(survey)
    s_tokens = _tokens(survey)

    # exact normalized SurveyId match first
    exact = [c for c in boundary_catalog if c["survey_norm"] == s_norm]

    if exact:
        best = exact[0]
    else:
        # fallback: token overlap score
        scored = []
        for c in boundary_catalog:
            inter = len(s_tokens & c["survey_tokens"])
            denom = max(1, len(s_tokens))
            score = inter / denom
            scored.append((score, c))
        scored.sort(key=lambda x: x[0], reverse=True)
        best = scored[0][1] if scored and scored[0][0] > 0 else None

    if best is None:
        print(f"No boundary found for: {survey}")
        continue

    boundary = best["boundary"]
    data_pairs[survey] = [
        sbes_match[0],
        mbes_match[0],
        boundary,
    ]

    print(f"{survey} -> {best['survey_id']} ({best['gdb_file'].name})")

# Bathymetric Raster Generation
- Entire survey, not just the small reach from the TN

In [ ]:
# specify the horizontal spatial resolution of the resulting bathymetric surface. This is in ft
# 10ft is being used here to match the resolution of the CSAT inputs
grid_res = 10.0

# True if you want the outputs of the interpolation methods will be showcased in this notebook using Matplotlib
plots = False    

# Set number of processors to use based on the number of CPUs your computer has
# NOTE: this may exceed your system memory depending on how much RAM you have
# My workstation has 24 CPUs and 96 GB of RAM; I will use between 8 and 12 processors
processors = 20

In [ ]:
# survey = list(data_pairs.keys())[3]
survey = "CL_02_UDS_20260305_CS_A"

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

OUT_DIR = WORK_DIR / survey
os.makedirs(OUT_DIR, exist_ok=True)

sbes_x = sbes_gdf.X_ft
sbes_y = sbes_gdf.Y_ft
sbes_z = sbes_gdf.Elevation_ft * -1

mbes_x = mbes_gdf.X_ft
mbes_y = mbes_gdf.Y_ft
mbes_z = mbes_gdf.Elevation_ft * -1

fig, ax = plt.subplots(1,2, figsize=(10, 5))

# Histogram for sbes_z
ax[0].hist(sbes_z, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[0].set_xlabel('Elevation (ft)', fontsize=12)
ax[0].set_ylabel('Frequency', fontsize=12)
ax[0].set_title(f'SBES Elevation Distribution (n={len(sbes_z):,})', fontsize=14)
ax[0].grid(axis='y', alpha=0.3)

# Histogram for mbes_z
ax[1].hist(mbes_z, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[1].set_xlabel('Elevation (ft)', fontsize=12)
ax[1].set_ylabel('Frequency', fontsize=12)
ax[1].set_title(f'MBES Elevation Distribution (n={len(mbes_z):,})', fontsize=14)
ax[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSBES Elevation Statistics (n={len(sbes_z):,}):")
print(f"  Mean: {np.mean(sbes_z):.2f} ft")
print(f"  Std: {np.std(sbes_z):.2f} ft")
print(f"  Min: {np.min(sbes_z):.2f} ft")
print(f"  Max: {np.max(sbes_z):.2f} ft")

print(f"\nMBES Elevation Statistics (n={len(mbes_z):,}):")
print(f"  Mean: {np.mean(mbes_z):.2f} ft")
print(f"  Std: {np.std(mbes_z):.2f} ft")
print(f"  Min: {np.min(mbes_z):.2f} ft")
print(f"  Max: {np.max(mbes_z):.2f} ft")

In [ ]:
# Multibeam

TINinterpolator = TIN(
    surveyId = survey,
    survey_path = OUT_DIR,
    x=mbes_x, y=mbes_y, z=mbes_z,
    survey_boundary = boundary,
    echosounder = "MBES",
    grid_res=grid_res,
    plot_outputs=plots,
    n_jobs=processors,
    max_edge_length=None
)

mbes_tin_path = TINinterpolator.generate()

In [ ]:
# All synthetic singlebeam

TINinterpolator = TIN(
    surveyId = survey,
    survey_path = OUT_DIR,
    x=sbes_x, y=sbes_y, z=sbes_z,
    survey_boundary = boundary,
    echosounder = "SBES",
    grid_res=grid_res,
    plot_outputs=plots,
    n_jobs=processors,
    max_edge_length=None
)

sbes_tin_path = TINinterpolator.generate()

In [ ]:
# Varying sample densities

point_spacings = [4, 8, 12, 16, 20, 50, 100]
trackline_spacings = [1, 2, 4, 8, 12, 16, 20]

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?


for itspace, tspace in enumerate(trackline_spacings):  # loop through the different transect spacings
    track_spacing = tspace * 25

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    sbes_gdf_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    mbes_gdf_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings):  # loop through the different point spacings

        along_length_temp = np.array(sbes_gdf_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = sbes_gdf_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        sbes_x = singlebeam_this_tspace_pspace.X_ft
        sbes_y = singlebeam_this_tspace_pspace.Y_ft
        sbes_z = singlebeam_this_tspace_pspace.Elevation_ft * -1


        TINinterpolator = TIN(
            surveyId = survey,
            survey_path = OUT_DIR,
            x=sbes_x, y=sbes_y, z=sbes_z,
            survey_boundary = boundary,
            echosounder = f"SBES_{track_spacing}_{pspace}",
            grid_res=grid_res,
            plot_outputs=plots,
            n_jobs=processors,
            max_edge_length=None
        )

        sbes_tin_path = TINinterpolator.generate()

# Generate bathymetric rasters for the small synthetic subset

In [ ]:
test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

transects_to_plot = list(range(380, 490))

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

sbes_x = sbes_gdf.X_ft
sbes_y = sbes_gdf.Y_ft
sbes_z = sbes_gdf.Elevation_ft

In [ ]:
TINinterpolator = TIN(
    surveyId = survey,
    survey_path = OUT_DIR,
    x=sbes_x, y=sbes_y, z=sbes_z,
    survey_boundary = boundary,
    echosounder = "clipped_SBES",
    grid_res=grid_res,
    plot_outputs=plots,
    n_jobs=processors,
    max_edge_length=None
)

sbes_tin_path = TINinterpolator.generate()

In [ ]:
# Varying sample densities

point_spacings = [4, 8, 12, 16, 20, 50, 100]
trackline_spacings = [1, 2, 4, 8, 12, 16, 20]

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

for itspace, tspace in enumerate(trackline_spacings):  # loop through the different transect spacings
    track_spacing = tspace * 25

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    sbes_gdf_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    mbes_gdf_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings):  # loop through the different point spacings

        along_length_temp = np.array(sbes_gdf_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = sbes_gdf_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        sbes_x = singlebeam_this_tspace_pspace.X_ft
        sbes_y = singlebeam_this_tspace_pspace.Y_ft
        sbes_z = singlebeam_this_tspace_pspace.Elevation_ft

        TINinterpolator = TIN(
            surveyId = survey,
            survey_path = OUT_DIR,
            x=sbes_x, y=sbes_y, z=sbes_z,
            survey_boundary = boundary,
            echosounder = f"clipped_SBES_{track_spacing}_{pspace}",
            grid_res=grid_res,
            plot_outputs=plots,
            n_jobs=processors,
            max_edge_length=None
        )

        sbes_tin_path = TINinterpolator.generate()

# Plot example of our sampling in UDS reach

In [ ]:
from scipy.spatial import Delaunay
from scipy.interpolate import LinearNDInterpolator

transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1] # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100] # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

In [ ]:
num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi = 300)

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        data_for_scipy = np.column_stack((singlebeam_this_tspace_pspace['X_ft'], singlebeam_this_tspace_pspace['Y_ft']))
        tri = Delaunay(data_for_scipy)
        interpolator = LinearNDInterpolator(tri, singlebeam_this_tspace_pspace['Elevation_ft'])
        interpolated_elevation = interpolator(multibeam_df_this_tspace['X_ft'], multibeam_df_this_tspace['Y_ft'])

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)

        # axes[itspace][ipspace].scatter(multibeam_df_this_tspace['X_ft'], multibeam_df_this_tspace['Y_ft'], s=0.2, c='black')
        im = axes[itspace][ipspace].scatter(multibeam_df_this_tspace['X_ft'], multibeam_df_this_tspace['Y_ft'], s=0.2,
                                       c=interpolated_elevation, zorder=1)
        for transect in transects_to_plot:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace][ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace][ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace][ipspace].set_aspect('equal')
        axes[itspace][ipspace].set_xticklabels([])
        axes[itspace][ipspace].set_yticklabels([])
        axes[itspace][ipspace].tick_params(left=False, bottom=False)
        # axes[itspace][ipspace].title('T=' + str(tspace*25) + ' ft, P=' + str(pspace) + ' ft')

        if ipspace == 0:
            axes[itspace][ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace][ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

        # print('debug')

fig.supxlabel('Point spacing on transect', fontsize=13, fontweight='bold')
fig.text(0.03, 0.5, 'Transect spacing', fontsize=13, fontweight='bold', rotation='vertical', va='center', ha='center')
plt.tight_layout()

fig.subplots_adjust(right=0.83)
cbar_ax = fig.add_axes([0.85, 0.15, 0.03, 0.5])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Elevation (ft MLLW)')
cbar.ax.tick_params(rotation=90)
plt.show()

# Calculate Dredge Cut Errors Relative to MBES surface and quantify cost differences caused by errors in sampling
- TODO: Use the same datasets to generate surfaces using other interpolation methods and compare differences

In [ ]:
mbes_dbm_path = Path('/mnt/teamgroup/AutoDBM/Public/CL_02_UDS_20260305_CS_A/MBES_TIN_bathy.tif')
clipped_sbes_dbm_paths = list(Path(f'/mnt/teamgroup/AutoDBM/Public/{survey}').glob('clipped_SBES*'))
tgt_ncf = ncf[ncf['channelareaidfk'].str.contains(f'{survey[:9]}')]

In [ ]:
test_mbes_dbm = xdem.DEM(mbes_dbm_path)

test_mbes_dbm.plot(
        cmap="viridis",
        vmin=float(test_mbes_dbm.get_stats("min")),
        vmax=float(test_mbes_dbm.get_stats("max")),
        cbar_title="Elevation (ft, MLLW)",
)

In [ ]:
# test_sbes_dbm = xdem.DEM(clipped_sbes_dbm_paths[2])
test_sbes_dbm = xdem.DEM(f'/mnt/teamgroup/AutoDBM/Public/{survey}/clipped_SBES_25_4_TIN_bathy.tif')

test_sbes_dbm.plot(
        cmap="viridis",
        vmin=float(test_sbes_dbm.get_stats("min")),
        vmax=float(test_sbes_dbm.get_stats("max")),
        cbar_title="Elevation (ft, MLLW)",
)

In [ ]:
clipped_mbes_dbm = test_mbes_dbm.reproject(test_sbes_dbm)

if isinstance(test_sbes_dbm.data, np.ma.MaskedArray):
    sbes_valid_mask = ~np.ma.getmaskarray(test_sbes_dbm.data)
else:
    sbes_valid_mask = ~np.isnan(test_sbes_dbm.data)

mbes_data = clipped_mbes_dbm.data.copy()

if isinstance(mbes_data, np.ma.MaskedArray):
    mbes_masked = np.ma.masked_where(~sbes_valid_mask, mbes_data)
else:
    mbes_masked = np.where(sbes_valid_mask, mbes_data, np.nan)

masked_mbes_dbm = clipped_mbes_dbm.copy()
masked_mbes_dbm.data = mbes_masked

masked_mbes_dbm.plot(
    cmap="viridis",
    vmin=float(np.nanmin(mbes_masked)),
    vmax=float(np.nanmax(mbes_masked)),
    cbar_title="Elevation (ft, MLLW)",
)

In [ ]:
ddbm=masked_mbes_dbm - test_sbes_dbm.reproject(clipped_mbes_dbm)
ddbm.plot(
        cmap="PuOr",
        vmin=float(ddbm.get_stats("min")),
        vmax=float(ddbm.get_stats("max")),
        cbar_title="Elevation differences (ft, MLLW)",
)

In [ ]:
price_per_cy = 9.29  # USD/CY dredged

dredge_cuts = {}

auth_depth = float(tgt_ncf["depthauthorized"].iloc[0])
elevation = masked_mbes_dbm.data

# Explicitly determine which cells have real data
if np.ma.is_masked(elevation):
    valid = ~np.ma.getmaskarray(elevation)
else:
    valid = np.isfinite(elevation)

elevation_filled = np.ma.filled(elevation, 0.0) if np.ma.is_masked(elevation) else elevation

# Compute cut = auth_depth + elevation (elevation is negative)
cut = auth_depth + elevation_filled
cut = np.where(cut > 0, cut, 0.0)
cut[~valid] = 0.0    # <-- zero out anything without real survey data

px, py = abs(masked_mbes_dbm.transform.a), abs(masked_mbes_dbm.transform.e)
volume_ft3 = np.sum(cut) * px * py
volume_cy = volume_ft3 / 27.0

dredge_cuts['MBES'] = (volume_cy, volume_cy * price_per_cy, masked_mbes_dbm)
print(volume_cy)

In [ ]:
for path in clipped_sbes_dbm_paths:
    dbm = xdem.DEM(path).reproject(masked_mbes_dbm)

    elevation = dbm.data

    # Explicitly determine which cells have real data
    if np.ma.is_masked(elevation):
        valid = ~np.ma.getmaskarray(elevation)
    else:
        valid = np.isfinite(elevation)

    elevation_filled = np.ma.filled(elevation, 0.0) if np.ma.is_masked(elevation) else elevation

    # Compute cut = auth_depth + elevation (elevation is negative)
    cut = auth_depth + elevation_filled
    cut = np.where(cut > 0, cut, 0.0)
    cut[~valid] = 0.0    # <-- zero out anything without real survey data

    px, py = abs(dbm.transform.a), abs(dbm.transform.e)
    volume_ft3 = np.sum(cut) * px * py
    volume_cy = volume_ft3 / 27.0

    dredge_cuts[path.stem[8:-10]] = (volume_cy, volume_cy * price_per_cy, dbm, masked_mbes_dbm - dbm) #masked_mbes_dbm - dbm

In [ ]:
transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1]  # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100]  # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

# --- Compute a global elevation range across all DEMs so the shared colorbar is correct ---
global_vmin = np.inf
global_vmax = -np.inf
for tspace in transect_spacings_to_plot:
    for pspace in point_spacings_to_plot:
        dem_key = f'SBES_{tspace * 25}_{pspace}'
        global_vmin = min(global_vmin, dredge_cuts[dem_key][2].get_stats('min'))
        global_vmax = max(global_vmax, dredge_cuts[dem_key][2].get_stats('max'))
del dem_key
# --------------------------------------------------------------------------------------

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi=300)
axes = np.atleast_2d(axes)  # ensures axes is always 2D, even if one dimension is 1

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        # --- Grab the pre-built DEM and plot the raster itself, using the shared global scale ---
        dem_key = f'SBES_{tspace * 25}_{pspace}'
        dem_this_tspace_pspace = dredge_cuts[dem_key][2]

        dem_this_tspace_pspace.plot(
            ax=axes[itspace, ipspace],
            cmap='viridis',
            add_cbar=False,
            vmin=global_vmin,
            vmax=global_vmax,
        )
        # -----------------------------------------------------------------------------------------

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot_local = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot_local)].reset_index(drop=True)

        for transect in transects_to_plot_local:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace, ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace, ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace, ipspace].set_aspect('equal')
        axes[itspace, ipspace].set_xticklabels([])
        axes[itspace, ipspace].set_yticklabels([])
        axes[itspace, ipspace].tick_params(left=False, bottom=False)

        if ipspace == 0:
            axes[itspace, ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace, ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

# --- Layout: run tight_layout first, THEN reserve margin and place sup-labels, so they aren't overridden ---
plt.tight_layout()
fig.subplots_adjust(right=0.83, bottom=0.20, wspace=0.08, hspace=0.08)
fig.supxlabel('Point spacing on transect', fontsize=15, fontweight='bold', y=0.12, x=0.45)
fig.canvas.draw()
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
grid_bottom = min(ax.get_position().y0 for ax in axes.flat)
grid_top = max(ax.get_position().y1 for ax in axes.flat)
fig.text(grid_left - 0.07, (grid_bottom + grid_top) / 2, 'Transect spacing', fontsize=15, fontweight='bold', rotation='vertical', va='center', ha='center')
# ------------------------------------------------------------------------------------------------------

# Use an explicit shared mappable so the colorbar is guaranteed to use the same global limits.
shared_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=global_vmin, vmax=global_vmax), cmap='viridis')
shared_mappable.set_array([])

cbar_ax = fig.add_axes([grid_right + 0.02, grid_bottom, 0.03, grid_top - grid_bottom])
cbar = fig.colorbar(shared_mappable, cax=cbar_ax)
cbar.set_label('Elevation (ft MLLW)', fontsize=13)
cbar.ax.tick_params(rotation=90)

fig.suptitle("TIN Prediction", fontsize=16, fontweight="bold", y=1.01)

plt.show()

In [ ]:
transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1]  # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100]  # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

# --- Compute a global elevation range across all DEMs so the shared colorbar is correct ---
global_vmin = np.inf
global_vmax = -np.inf
for tspace in transect_spacings_to_plot:
    for pspace in point_spacings_to_plot:
        dem_key = f'SBES_{tspace * 25}_{pspace}'
        global_vmin = min(global_vmin, dredge_cuts[dem_key][3].get_stats('min'))
        global_vmax = max(global_vmax, dredge_cuts[dem_key][3].get_stats('max'))
del dem_key
# --------------------------------------------------------------------------------------

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi=300)
axes = np.atleast_2d(axes)  # ensures axes is always 2D, even if one dimension is 1

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        # --- Grab the pre-built DEM and plot the raster itself, using the shared global scale ---
        dem_key = f'SBES_{tspace * 25}_{pspace}'
        dem_this_tspace_pspace = dredge_cuts[dem_key][3]

        dem_this_tspace_pspace.plot(
            ax=axes[itspace, ipspace],
            cmap='PuOr',
            add_cbar=False,
            vmin=global_vmin,
            vmax=global_vmax,
        )
        # -----------------------------------------------------------------------------------------

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot_local = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot_local)].reset_index(drop=True)

        for transect in transects_to_plot_local:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace, ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace, ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace, ipspace].set_aspect('equal')
        axes[itspace, ipspace].set_xticklabels([])
        axes[itspace, ipspace].set_yticklabels([])
        axes[itspace, ipspace].tick_params(left=False, bottom=False)

        if ipspace == 0:
            axes[itspace, ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace, ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

# --- Layout: run tight_layout first, THEN reserve margin and place sup-labels, so they aren't overridden ---
plt.tight_layout()
fig.subplots_adjust(right=0.83, bottom=0.20, wspace=0.08, hspace=0.08)
fig.supxlabel('Point spacing on transect', fontsize=15, fontweight='bold', y=0.12, x=0.45)
fig.canvas.draw()
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
grid_bottom = min(ax.get_position().y0 for ax in axes.flat)
grid_top = max(ax.get_position().y1 for ax in axes.flat)
fig.text(grid_left - 0.07, (grid_bottom + grid_top) / 2, 'Transect spacing', fontsize=15, fontweight='bold', rotation='vertical', va='center', ha='center')
# ------------------------------------------------------------------------------------------------------

# Use an explicit shared mappable so the colorbar is guaranteed to use the same global limits.
shared_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=global_vmin, vmax=global_vmax), cmap='PuOr')
shared_mappable.set_array([])

cbar_ax = fig.add_axes([grid_right + 0.02, grid_bottom, 0.03, grid_top - grid_bottom])
cbar = fig.colorbar(shared_mappable, cax=cbar_ax)
cbar.set_label('Error (ft)', fontsize=13)
cbar.ax.tick_params(rotation=90)

fig.suptitle("TIN Error", fontsize=16, fontweight="bold", y=1.01)

plt.show()

In [ ]:
base_cut = dredge_cuts['MBES'][0]
base_price = dredge_cuts['MBES'][1]

diffs = []
for k, (vol, price, _) in dredge_cuts.items():
    if k == 'MBES' or k=='SBES':
        continue
    else:
        _, ts, ss = k.split("_")
        # vol_pct_diff = ((base_cut - vol) / ((base_cut + vol)/2))*100
        # price_pct_diff = ((base_price - price) / ((base_price + price)/2))*100
        vol_pct_change = ((base_cut - vol) / base_cut)*100
        price_pct_change = ((base_price - price) / base_price)*100

        # vol_diff = base_cut - vol
        # price_diff = base_price - price

        diffs.append({
            'Track_Spacing': ts,
            'Sounding_Spacing': ss,
            'Vol_pct_Diff': vol_pct_change,
            'Cost_pct_Diff': price_pct_change
        })

df = pd.DataFrame(diffs)

In [ ]:
sampled_df = df[df['Track_Spacing'].isin(['300', '100', '25'])]
sampled_df = sampled_df[sampled_df['Sounding_Spacing'].isin(['4', '20', '100'])]
sampled_df

In [ ]:
sampled_df['Track_Spacing'] = sampled_df['Track_Spacing'].astype(float)
sampled_df['Sounding_Spacing'] = sampled_df['Sounding_Spacing'].astype(float)

track_spacing = sorted(sampled_df['Track_Spacing'].unique())
sounding_spacing = sorted(sampled_df['Sounding_Spacing'].unique())

fig, axes = plt.subplots(2, 1, figsize=(13, 9), dpi=300)
colors = plt.cm.viridis(np.linspace(0, 1, len(sounding_spacing)))
n = len(sounding_spacing)
width = 0.8 / n
x = np.arange(len(track_spacing))

baseline_labels = {
    'Vol_Diff': f'Baseline Volume: {base_cut:,.0f} CY',
    'Cost_Diff': f'Baseline Cost: ${base_price:,.0f}',
}

for ax, col, title, ylabel in zip(
        axes, ['Vol_Diff', 'Cost_Diff'],
        ['Dredge Cut Volume Difference', f'Estimated Cost Difference (at ${price_per_cy}/CY)'],
        ['ΔVolume (CY)', 'ΔCost ($)']):
    for i, (ss, c) in enumerate(zip(sounding_spacing, colors)):
        sub = sampled_df[sampled_df['Sounding_Spacing'] == ss].set_index('Track_Spacing')
        vals = sub.reindex(track_spacing)[col].values
        ax.bar(x + (i - n / 2) * width + width / 2, vals, width=width, color=c,
               label=f'{int(ss)} ft', edgecolor='white', linewidth=0.3)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{int(t)}' for t in track_spacing], fontsize=13)
    ax.tick_params(axis='y', labelsize=13)
    ax.set_xlabel('Transect spacing (ft)', fontweight='bold', fontsize=15)
    ax.set_ylabel(ylabel, fontweight='bold', fontsize=15)
    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')

    # baseline_key = col.replace('_Abs', '')
    ax.text(
        0.02, 0.1, baseline_labels[col],
        transform=ax.transAxes,
        fontsize=13, fontweight='bold', va='top', ha='left',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor='black', alpha=0.85)
    )

leg = axes[0].legend(
    title='Point spacing',
    bbox_to_anchor=(1.01, 1),
    loc='upper left',
    prop={'size': 13, 'weight': 'bold'},
    title_fontsize=13
)
leg.get_title().set_weight('bold')
plt.tight_layout()
# plt.savefig('/mnt/user-data/outputs/bar_chart.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
sampled_df['Track_Spacing'] = sampled_df['Track_Spacing'].astype(float)
sampled_df['Sounding_Spacing'] = sampled_df['Sounding_Spacing'].astype(float)

track_spacing = sorted(sampled_df['Track_Spacing'].unique())
sounding_spacing = sorted(sampled_df['Sounding_Spacing'].unique())

fig, axes = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
colors = plt.cm.viridis(np.linspace(0, 1, len(sounding_spacing)))
n = len(sounding_spacing)
width = 0.8 / n
x = np.arange(len(track_spacing))

baseline_labels = {
    'Vol_pct_Diff': f'Baseline Volume: {base_cut:,.0f} CY',
    'Vol_pct_Diff': f'Baseline Volume: {base_cut:,.0f} CY',
}

ax = axes
col = 'Vol_pct_Diff'
title = 'Dredge Estimate Changes'
ylabel = '% Change in \nDredge Estimates (CY, $/CY)'

for i, (ss, c) in enumerate(zip(sounding_spacing, colors)):
    sub = sampled_df[sampled_df['Sounding_Spacing'] == ss].set_index('Track_Spacing')
    vals = sub.reindex(track_spacing)[col].values
    ax.bar(
        x + (i - n / 2) * width + width / 2,
        vals,
        width=width,
        color=c,
        label=f'{int(ss)} ft',
        edgecolor='white',
        linewidth=0.3,
    )

ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels([f'{int(t)}' for t in track_spacing], fontsize=13)
ax.tick_params(axis='y', labelsize=13)
ax.set_xlabel('Transect spacing (ft)', fontweight='bold', fontsize=15)
ax.set_ylabel(ylabel, fontweight='bold', fontsize=15)
ax.set_title(title, fontsize=15, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

ax.text(
    0.02, 0.1, f'Baseline Volume: {base_cut:,.0f} CY',
    transform=ax.transAxes,
    fontsize=13, fontweight='bold', va='top', ha='left',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
              edgecolor='black', alpha=0.85)
)

ax.text(
    0.02, 0.2, f'Baseline Price: ${base_price:,.0f}',
    transform=ax.transAxes,
    fontsize=13, fontweight='bold', va='top', ha='left',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
              edgecolor='black', alpha=0.85)
)

leg = ax.legend(
    title='Point spacing',
    bbox_to_anchor=(1.01, 1),
    loc='upper left',
    prop={'size': 13, 'weight': 'bold'},
    title_fontsize=13
)
leg.get_title().set_weight('bold')
plt.tight_layout()
# plt.savefig('/mnt/user-data/outputs/bar_chart.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
sampled_df['Track_Spacing'] = sampled_df['Track_Spacing'].astype(float)
sampled_df['Sounding_Spacing'] = sampled_df['Sounding_Spacing'].astype(float)

track_spacing = sorted(sampled_df['Track_Spacing'].unique())
sounding_spacing = sorted(sampled_df['Sounding_Spacing'].unique())

fig, axes = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
colors = plt.cm.viridis(np.linspace(0, 1, len(sounding_spacing)))
n = len(sounding_spacing)
width = 0.8 / n
x = np.arange(len(track_spacing))

baseline_labels = {
    'Vol_Diff': f'Baseline Volume: {base_cut:,.0f} CY',
}

ax = axes
col = 'Vol_Diff'
title = 'Dredge Cut Volume Difference'
ylabel = 'ΔVolume (CY)'

for i, (ss, c) in enumerate(zip(sounding_spacing, colors)):
    sub = sampled_df[sampled_df['Sounding_Spacing'] == ss].set_index('Track_Spacing')
    vals = sub.reindex(track_spacing)[col].values
    ax.bar(
        x + (i - n / 2) * width + width / 2,
        vals,
        width=width,
        color=c,
        label=f'{int(ss)} ft',
        edgecolor='white',
        linewidth=0.3,
    )

ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels([f'{int(t)}' for t in track_spacing], fontsize=13)
ax.tick_params(axis='y', labelsize=13)
ax.set_xlabel('Transect spacing (ft)', fontweight='bold', fontsize=15)
ax.set_ylabel(ylabel, fontweight='bold', fontsize=15)
ax.set_title(title, fontsize=15, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

ax.text(
    0.02, 0.1, baseline_labels[col],
    transform=ax.transAxes,
    fontsize=13, fontweight='bold', va='top', ha='left',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
              edgecolor='black', alpha=0.85)
)

leg = ax.legend(
    title='Point spacing',
    bbox_to_anchor=(1.01, 1),
    loc='upper left',
    prop={'size': 13, 'weight': 'bold'},
    title_fontsize=13
)
leg.get_title().set_weight('bold')
plt.tight_layout()
# plt.savefig('/mnt/user-data/outputs/bar_chart.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
sampled_df['Track_Spacing'] = sampled_df['Track_Spacing'].astype(float)
sampled_df['Sounding_Spacing'] = sampled_df['Sounding_Spacing'].astype(float)

sampled_df['Vol_Diff_Abs'] = sampled_df['Vol_Diff'].abs()
sampled_df['Cost_Diff_Abs'] = sampled_df['Cost_Diff'].abs()

track_spacing = sorted(sampled_df['Track_Spacing'].unique())
sounding_spacing = sorted(sampled_df['Sounding_Spacing'].unique())

fig, axes = plt.subplots(2, 1, figsize=(13, 9), dpi=300)
colors = plt.cm.viridis(np.linspace(0, 1, len(sounding_spacing)))
n = len(sounding_spacing)
width = 0.8 / n
x = np.arange(len(track_spacing))

baseline_labels = {
    'Vol_Diff': f'Baseline Volume: {base_cut:,.0f} CY',
    'Cost_Diff': f'Baseline Cost: ${base_price:,.0f}',
}

for ax, col, title, ylabel in zip(
        axes, ['Vol_Diff_Abs', 'Cost_Diff_Abs'],
        ['Dredge Cut Volume Absolute Difference', 'Estimated Cost Absolute Difference (at $25/CY)'],
        ['|Volume Diff| (CY)', '|Cost Diff| ($)']):
    for i, (ss, c) in enumerate(zip(sounding_spacing, colors)):
        sub = sampled_df[sampled_df['Sounding_Spacing'] == ss].set_index('Track_Spacing')
        vals = sub.reindex(track_spacing)[col].values
        ax.bar(x + (i - n / 2) * width + width / 2, vals, width=width, color=c,
               label=f'{int(ss)} ft', edgecolor='white', linewidth=0.3)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{int(t)}' for t in track_spacing], fontsize=13)
    ax.tick_params(axis='y', labelsize=13)
    ax.set_xlabel('Transect spacing (ft)', fontweight='bold', fontsize=15)
    ax.set_ylabel(ylabel, fontweight='bold', fontsize=15)
    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')

    baseline_key = col.replace('_Abs', '')
    ax.text(
        0.02, 0.96, baseline_labels[baseline_key],
        transform=ax.transAxes,
        fontsize=13, fontweight='bold', va='top', ha='left',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor='black', alpha=0.85)
    )

leg = axes[0].legend(
    title='Point spacing',
    bbox_to_anchor=(1.01, 1),
    loc='upper left',
    prop={'size': 13, 'weight': 'bold'},
    title_fontsize=13
)
leg.get_title().set_weight('bold')
plt.tight_layout()
# plt.savefig('/mnt/user-data/outputs/bar_chart.png', dpi=300, bbox_inches='tight')
plt.show()

# Maybe try something similar with other interpolation methods?
- Greater than 10k points -> GP Regression
- Less than 10k points -> Kriging

In [ ]:
transects_to_plot = list(range(380, 490))

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

In [ ]:
# GP for >10k points
# Varying sample densities

point_spacings = [100]    # 4, 20, 100
trackline_spacings = [12]    # 1, 4, 12

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?


for itspace, tspace in enumerate(trackline_spacings):  # loop through the different transect spacings
    track_spacing = tspace * 25

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    sbes_gdf_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    mbes_gdf_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings):  # loop through the different point spacings

        along_length_temp = np.array(sbes_gdf_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = sbes_gdf_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        sbes_x = singlebeam_this_tspace_pspace.X_ft
        sbes_y = singlebeam_this_tspace_pspace.Y_ft
        sbes_z = singlebeam_this_tspace_pspace.Elevation_ft

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 5))

sbes_z_norm = (sbes_z - np.mean(sbes_z)) / np.std(sbes_z)

# Histogram for sbes_z_norm
ax.hist(sbes_z_norm, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Standardized Elevation', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title(f'SBES Standardized Elevation Distribution (n={len(sbes_z_norm):,})', fontsize=14)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSBES Elevation Statistics (n={len(sbes_z_norm):,}):")
print(f"  Mean: {np.mean(sbes_z_norm):.2f} ft")
print(f"  Std: {np.std(sbes_z_norm):.2f} ft")
print(f"  Min: {np.min(sbes_z_norm):.2f} ft")
print(f"  Max: {np.max(sbes_z_norm):.2f} ft")

In [ ]:
import verde as vd

sbes_xy_raw = np.column_stack([sbes_x, sbes_y])

sbes_chain = vd.Chain([
    ("reduce", vd.BlockMean(spacing=50)),   # adjust spacing to your CRS units
    ("spline", vd.Spline(damping=1e-4)),
]).fit((sbes_xy_raw[:, 0], sbes_xy_raw[:, 1]), sbes_z)

# chain.fit((sbes_xy_raw[:, 0], sbes_xy_raw[:, 1]), sbes_z)
sbes_trend_pred = sbes_chain.predict((sbes_xy_raw[:, 0], sbes_xy_raw[:, 1]))   # evaluate at full resolution
sbes_residuals = sbes_z - sbes_trend_pred

fig, ax = plt.subplots(1, 1, figsize=(20, 5))

# Histogram for sbes_residuals
ax.hist(sbes_residuals, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Detrended Elevation', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title(f'SBES Trend Elevation Distribution (n={len(sbes_residuals):,})', fontsize=14)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSBES Detrended Elevation Statistics (n={len(sbes_residuals):,}):")
print(f"  Mean: {np.mean(sbes_residuals):.2f} ft")
print(f"  Std: {np.std(sbes_residuals):.2f} ft")
print(f"  Min: {np.min(sbes_residuals):.2f} ft")
print(f"  Max: {np.max(sbes_residuals):.2f} ft")

In [ ]:
# normalize the detrended data

sbes_residuals_norm = (sbes_residuals - np.mean(sbes_residuals)) / np.std(sbes_residuals)

fig, ax = plt.subplots(1, 1, figsize=(20, 5))

# Histogram for sbes_residuals_norm
ax.hist(sbes_residuals_norm, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Standardized Detrended Elevation', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title(f'SBES Standardized Trend Elevation Distribution (n={len(sbes_residuals_norm):,})', fontsize=14)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSBES Detrended Elevation Statistics (n={len(sbes_residuals_norm):,}):")
print(f"  Mean: {np.mean(sbes_residuals_norm):.2f} ft")
print(f"  Std: {np.std(sbes_residuals_norm):.2f} ft")
print(f"  Min: {np.min(sbes_residuals_norm):.2f} ft")
print(f"  Max: {np.max(sbes_residuals_norm):.2f} ft")

In [ ]:
import torch
import gpytorch
from torch.utils.data import TensorDataset, DataLoader

from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution
from gpytorch.variational import VariationalStrategy
from gpytorch.constraints import Interval, GreaterThan, LessThan
from gpytorch.priors import GammaPrior, LogNormalPrior, UniformPrior, NormalPrior
from gpytorch.means import Mean

from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from rasterio.transform import from_origin

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# train on all sbes points
X_all = torch.as_tensor(sbes_xy_raw, dtype=torch.float32)
y_all = torch.as_tensor(sbes_residuals_norm, dtype=torch.float32)

# Standardize the raw coordinates as they are in a different scale than the elevation
x_mean = X_all.mean(0)
x_std = X_all.std(0).clamp_min(1e-6)
X_all_scaled = (X_all - x_mean) / x_std

if torch.cuda.is_available():
    X_all_scaled = X_all_scaled.cuda().float()
    y_all = y_all.cuda().float()
sbes_xy_raw = np.asarray(sbes_xy_raw, dtype=np.float32)
x_min, x_max = float(sbes_xy_raw[:, 0].min()), float(sbes_xy_raw[:, 0].max())
y_min, y_max = float(sbes_xy_raw[:, 1].min()), float(sbes_xy_raw[:, 1].max())

nx = int(np.ceil((x_max - x_min) / grid_res)) + 1
ny = int(np.ceil((y_max - y_min) / grid_res)) + 1
x_grid = np.linspace(x_min, x_min + (nx - 1) * grid_res, nx, dtype=np.float32)
y_grid = np.linspace(y_min, y_min + (ny - 1) * grid_res, ny, dtype=np.float32)

transform = from_origin(
    x_min - grid_res / 2,
    y_max + grid_res / 2,
    grid_res, grid_res
)

grid_xx, grid_yy = np.meshgrid(x_grid, y_grid)  # shape (ny, nx) by default
grid_points = torch.stack([
    torch.tensor(grid_xx.flatten(), dtype=torch.float32),
    torch.tensor(grid_yy.flatten(), dtype=torch.float32)
], dim=1)

grid_points_s = (grid_points - x_mean.cpu()) / x_std.cpu()

In [ ]:
all_train_dataset = TensorDataset(X_all_scaled, y_all)
# all_train_loader = DataLoader(all_train_dataset, batch_size=1024, shuffle=True)\

all_train_loader = DataLoader(
    all_train_dataset,
    batch_size=2048, 
    shuffle=True,
    num_workers=0 # Keep at 0 since data is already on GPU
)

In [ ]:
class GPModel(ApproximateGP):
    def __init__(self, inducing_points, train_x, train_z):
        variational_distribution = CholeskyVariationalDistribution(inducing_points.size(0))
        variational_strategy = VariationalStrategy(
            self, inducing_points, variational_distribution,
            learn_inducing_locations=True,
        )
        super().__init__(variational_strategy)

        # TODO: Try out different mean functions
        # Maybe even try some of the means with first and second derivatives?
        
        self.mean_module = gpytorch.means.ZeroMean()
        # self.mean_module = gpytorch.means.LinearMean(input_size=2)

        ext = (train_x.max(dim=0).values - train_x.min(dim=0).values).detach().cpu()
        dmin, dmax = float(ext.min()), float(ext.max())

        self.covar_short = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.MaternKernel(
                nu=1.5, ard_num_dims=2,
                lengthscale_constraint=Interval(dmin * 0.005, dmin * 0.10),
            ),
            outputscale_prior=GammaPrior(2.0, 4.0),
        )
        self.covar_long = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.MaternKernel(
                nu=2.5, ard_num_dims=2,
                lengthscale_constraint=Interval(dmin * 0.10, dmax * 0.50),  
            ),
            outputscale_prior=GammaPrior(2.0, 4.0),
        )

        self.covar_short.base_kernel.lengthscale = dmin * 0.03
        self.covar_long.base_kernel.lengthscale = dmin * 0.20
        self.covar_short.outputscale = 0.1
        self.covar_long.outputscale = 0.1

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_short(x) + self.covar_long(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

In [ ]:
from sklearn.cluster import MiniBatchKMeans

X_unique = torch.unique(X_all_scaled, dim=0)

# TODO: refine this so that if there are less than 3000 points, an exact GP is used instead of an SVGP
# TODO: also, maybe have it increase for huge datasets. 

def init_inducing_points(X, n_inducing=3000):
    # Use MiniBatchKMeans on CPU for speed, then move back to the active device
    kmeans = MiniBatchKMeans(n_clusters=n_inducing, batch_size=10000, n_init=3, random_state=42)
    kmeans.fit(X.detach().cpu().numpy())
    centers = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32)

    # Tiny perturbation helps avoid near-duplicate inducing locations that can destabilize covariance matrix
    centers = centers + 1e-5 * torch.randn_like(centers)
    return centers.to(X.device)

num_inducing = min(5000, X_unique.size(0))
inducing_points = init_inducing_points(X_unique, num_inducing)

In [ ]:
train_x_all, train_y_all = all_train_loader.dataset.tensors

model = GPModel(inducing_points=inducing_points, train_x=train_x_all, train_z=train_y_all)

likelihood = gpytorch.likelihoods.GaussianLikelihood(
    noise_constraint=gpytorch.constraints.GreaterThan(1e-3),
    noise_prior=GammaPrior(2.0, 50.0),
)

if torch.cuda.is_available():
    model = model.cuda()
    likelihood = likelihood.cuda()

# Explicit, non-fragile parameter split.
variational_optimizer = gpytorch.optim.NGD(
    model.variational_parameters(),
    num_data=train_y_all.size(0),
    lr=0.01,
)
hyperparameter_optimizer = torch.optim.Adam(
    [{"params": model.hyperparameters()}, {"params": likelihood.parameters()}],
    lr=0.01,
)

mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=train_y_all.size(0))

In [ ]:
import tqdm
num_epochs = 50

model.train()
likelihood.train()

optimizer = torch.optim.Adam([
    {'params': model.parameters()},
    {'params': likelihood.parameters()},
], lr=0.01)

# Our loss object. We're using the VariationalELBO for variational inference
mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=y_all.size(0))

epochs_iter = tqdm.notebook.tqdm(range(num_epochs), desc="Epoch")
with gpytorch.settings.cholesky_jitter(1e-4), gpytorch.settings.variational_cholesky_jitter(1e-4), gpytorch.settings.cholesky_max_tries(10):
    for i in epochs_iter:
        # Within each iteration, we will go over each minibatch of data
        minibatch_iter = tqdm.notebook.tqdm(all_train_loader, desc="Minibatch", leave=False)
        for x_batch, y_batch in minibatch_iter:
            optimizer.zero_grad()
            output = model(x_batch)
            loss = -mll(output, y_batch)
            minibatch_iter.set_postfix(loss=loss.item())
            loss.backward()
            optimizer.step()

In [ ]:
from shapely import contains_xy

model.eval()
likelihood.eval()
pred_means = []
pred_stds = []

with torch.no_grad(), gpytorch.settings.fast_pred_var():
    for i in range(0, grid_points_s.size(0), 2048):
        batch = grid_points_s[i:i+2048].cuda() if torch.cuda.is_available() else grid_points_s[i:i+2048]
        # preds = likelihood(model(batch))
        preds = model(batch)
        pred_means.append(preds.mean.cpu())
        pred_stds.append(preds.stddev.cpu())

pred_means = torch.cat(pred_means)
pred_stds = torch.cat(pred_stds)

# Convert back to real units
z_grid = pred_means * np.std(sbes_residuals_norm) + np.mean(sbes_residuals_norm)
z_uncertainty = pred_stds * np.std(sbes_residuals_norm)

trend_pred_grid = sbes_chain.predict((grid_points[:, 0].numpy(), grid_points[:, 1].numpy()))
z_grid_true = z_grid.numpy() + trend_pred_grid

# Reshape to 2D grid'
z_surface = z_grid_true.reshape(grid_xx.shape)
z_unc_surface = z_uncertainty.reshape(grid_xx.shape)

grid_np = grid_points.numpy()
mask = contains_xy((boundary).union_all(), grid_np[:, 0], grid_np[:, 1])

# Apply mask to predictions.
z_grid_masked = z_grid_true
z_unc_masked = z_uncertainty
z_grid_masked[~mask] = np.nan
z_unc_masked[~mask] = np.nan

# Reshape back to 2D arrays aligned with transform/from_origin.
z_surface_masked = z_grid_masked.reshape(grid_xx.shape)
z_unc_surface_masked = z_unc_masked.reshape(grid_xx.shape)
 
z_raster = np.flipud(z_surface_masked).astype(np.float32)
z_unc_raster = np.flipud(z_unc_surface_masked).astype(np.float32)

In [ ]:
import contextily as cx
from rasterio.warp import transform_bounds

ncf_polygons_3857 = ncf.to_crs(epsg=3857)

# Reproject raster extent from survey CRS to Web Mercator.
x_min_3857, y_min_3857, x_max_3857, y_max_3857 = transform_bounds(
    data_crs, "EPSG:3857", x_min, y_min, x_max, y_max
)
extent_3857 = [x_min_3857, x_max_3857, y_min_3857, y_max_3857]

# plot predictions and prediction uncertainty
fig, axes = plt.subplots(1, 2, figsize=(18, 9), dpi=150)

im1 = axes[0].imshow(
    np.flipud(z_raster), origin="lower", extent=extent_3857,
    cmap="viridis", alpha=0.7, zorder=2
)

ncf_polygons_3857.plot(ax=axes[0], facecolor='none', edgecolor='red',
                       linewidth=2, zorder=3)
cx.add_basemap(axes[0], source=cx.providers.OpenStreetMap.Mapnik,
               crs="EPSG:3857", zorder=1)
axes[0].set_xlim(extent_3857[0], extent_3857[1])
axes[0].set_ylim(extent_3857[2], extent_3857[3])
axes[0].set_title("SVGP Interpolation (Depth)")
axes[0].set_xlabel("Easting (m, Web Mercator)")
axes[0].set_ylabel("Northing (m, Web Mercator)")
fig.colorbar(im1, ax=axes[0], label="Depth (ft)", shrink=0.8)


im2 = axes[1].imshow(
    np.flipud(z_unc_raster), origin="lower", extent=extent_3857,
    cmap="magma_r", alpha=0.7, zorder=2
)

ncf_polygons_3857.plot(ax=axes[1], facecolor='none', edgecolor='red',
                       linewidth=2, zorder=3)
cx.add_basemap(axes[1], source=cx.providers.OpenStreetMap.Mapnik,
               crs="EPSG:3857", zorder=1)
axes[1].set_xlim(extent_3857[0], extent_3857[1])
axes[1].set_ylim(extent_3857[2], extent_3857[3])
axes[1].set_title("SVGP Predictive Uncertainty (1σ)")
axes[1].set_xlabel("Easting (m, Web Mercator)")
axes[1].set_ylabel("Northing (m, Web Mercator)")
fig.colorbar(im2, ax=axes[1], label="Uncertainty (ft)", shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
import rasterio

meta = {
    'driver': 'GTiff',
    'height': z_raster.shape[0],
    'width': z_raster.shape[1],
    'count': 1,
    'dtype': 'float32',
    'crs': data_crs,
    'transform': transform,
    'nodata': -9999,
    'tiled': True,
    'blockxsize': 512,
    'blockysize': 512,
    'compress': 'deflate',
    'predictor': 3,
    'zlevel': 6
}

with rasterio.open(OUT_DIR / f'clipped_{track_spacing}_{pspace}_SVGP_bathy.tif', 'w', **meta) as dst:
    dst.write(z_raster, 1)
    dst.build_overviews([2, 4, 8, 16, 32], rasterio.enums.Resampling.average)

with rasterio.open(OUT_DIR / f'clipped_{track_spacing}_{pspace}_SVGP_uncertainty.tif', 'w', **meta) as dst:
    dst.write(z_unc_raster, 1)
    dst.build_overviews([2, 4, 8, 16, 32], rasterio.enums.Resampling.average)

# make sure to run the cross-validation scheme like the deterministic methods

# df , stats = processing_help.validate_bathymetric_surface(OUT_DIR / 'SVGP_bathy.tif', mbes_x, mbes_y, mbes_z)

In [ ]:
OUT_DIR

In [ ]:
gp_bathy_path = OUT_DIR / f'clipped_{track_spacing}_{pspace}_SVGP_bathy.tif'
gp_unc_path = OUT_DIR / f'clipped_{track_spacing}_{pspace}_SVGP_uncertainty.tif'

In [ ]:
dredge_cuts[f"SBES_{track_spacing}_{pspace}"][-1].plot(
    cmap="viridis",
    vmin=float(dredge_cuts[f"SBES_{track_spacing}_{pspace}"][-1].get_stats("min")),
    vmax=float(dredge_cuts[f"SBES_{track_spacing}_{pspace}"][-1].get_stats("max")),
    cbar_title="Elevation (ft, MLLW)",
)

In [ ]:
gp_dbm = xdem.DEM(gp_bathy_path)

clipped_gp_dbm = gp_dbm.reproject(masked_mbes_dbm)

if isinstance(masked_mbes_dbm.data, np.ma.MaskedArray):
    mbes_valid_mask = ~np.ma.getmaskarray(masked_mbes_dbm.data)
else:
    mbes_valid_mask = ~np.isnan(masked_mbes_dbm.data)

gp_data = clipped_gp_dbm.data.copy()

if isinstance(gp_data, np.ma.MaskedArray):
    gp_masked = np.ma.masked_where(~mbes_valid_mask, gp_data)
else:
    gp_masked = np.where(mbes_valid_mask, gp_data, np.nan)

masked_gp_dbm = clipped_gp_dbm.copy()
masked_gp_dbm.data = gp_masked

masked_gp_dbm.plot(
    cmap="viridis",
    vmin=float(masked_gp_dbm.get_stats("min")),
    vmax=float(masked_gp_dbm.get_stats("max")),
    cbar_title="Elevation (ft, MLLW)",
)

In [ ]:
gp_ddbm=masked_mbes_dbm - masked_gp_dbm
gp_ddbm.plot(
        cmap="PuOr",
        vmin=float(gp_ddbm.get_stats("min")),
        vmax=float(gp_ddbm.get_stats("max")),
        cbar_title="Elevation differences (ft, MLLW)",
)

In [ ]:
unc_obj = xdem.DEM(gp_unc_path)

clipped_gp_unc_dbm = unc_obj.reproject(masked_mbes_dbm)

if isinstance(masked_mbes_dbm.data, np.ma.MaskedArray):
    mbes_valid_mask = ~np.ma.getmaskarray(masked_mbes_dbm.data)
else:
    mbes_valid_mask = ~np.isnan(masked_mbes_dbm.data)

gp_unc_data = clipped_gp_unc_dbm.data.copy()

if isinstance(gp_unc_data, np.ma.MaskedArray):
    gp_unc_masked = np.ma.masked_where(~mbes_valid_mask, gp_unc_data)
else:
    gp_unc_masked = np.where(mbes_valid_mask, gp_unc_data, np.nan)

masked_unc_obj = clipped_gp_unc_dbm.copy()
masked_unc_obj.data = gp_unc_masked

In [ ]:
raw_data = masked_gp_dbm.data
if isinstance(raw_data, np.ma.MaskedArray):
    nodata_mask = np.ma.getmaskarray(raw_data)
    elevation = raw_data.filled(np.nan)
else:
    elevation = np.asarray(raw_data, dtype=float)
    nodata_mask = np.isnan(elevation)

raw_unc = masked_unc_obj.data
if isinstance(raw_unc, np.ma.MaskedArray):
    nodata_mask = np.ma.getmaskarray(raw_unc)
    unc = raw_unc.filled(np.nan)
else:
    unc = np.asarray(raw_unc, dtype=float)
    nodata_mask = np.isnan(unc)

# Compute cut = auth_depth + elevation (elevation is negative)
cut = auth_depth + elevation
cut = np.where(cut > 0, cut, 0.0)
cut[nodata_mask] = 0.0

cut_up = cut + (1.96*unc)
cut_up = np.where(cut_up > 0, cut_up, 0.0)
cut_up[nodata_mask] = 0.0

cut_lo = cut - (1.96*unc)
cut_lo = np.where(cut_lo > 0, cut_lo, 0.0)
cut_lo[nodata_mask] = 0.0

px, py = abs(masked_gp_dbm.transform.a), abs(masked_gp_dbm.transform.e)
volume_ft3 = np.sum(cut) * px * py
volume_ft3_up = np.sum(cut_up) * px * py
volume_ft3_lo = np.sum(cut_lo) * px * py
volume_cy = volume_ft3 / 27.0
volume_cy_up = volume_ft3_up / 27.0
volume_cy_lo = volume_ft3_lo / 27.0

print(volume_cy, volume_cy*price_per_cy)
print(volume_cy_up, volume_cy_up*price_per_cy)
print(volume_cy_lo, volume_cy_lo*price_per_cy)

In [ ]:
print(dredge_cuts[f'SBES_{track_spacing}_{pspace}'][0], dredge_cuts[f'SBES_{track_spacing}_{pspace}'][1])

In [ ]:
gp_bathy_paths = list(OUT_DIR.glob('clipped_*SVGP*bathy.tif'))
gp_dredge_cuts = {}

for path in gp_bathy_paths:
    dbm = xdem.DEM(path).reproject(masked_mbes_dbm)
    unc_dbm = xdem.DEM(str(path).replace('bathy', 'uncertainty')).reproject(masked_mbes_dbm)


    if isinstance(masked_mbes_dbm.data, np.ma.MaskedArray):
        mbes_valid_mask = ~np.ma.getmaskarray(masked_mbes_dbm.data)
    else:
        mbes_valid_mask = ~np.isnan(masked_mbes_dbm.data)

    gp_data = dbm.data.copy()

    if isinstance(gp_data, np.ma.MaskedArray):
        gp_masked = np.ma.masked_where(~mbes_valid_mask, gp_data)
    else:
        gp_masked = np.where(mbes_valid_mask, gp_data, np.nan)

    masked_gp_dbm = dbm.copy()
    masked_gp_dbm.data = gp_masked

    gp_unc_data = unc_dbm.data.copy()

    if isinstance(gp_unc_data, np.ma.MaskedArray):
        gp_unc_masked = np.ma.masked_where(~mbes_valid_mask, gp_unc_data)
    else:
        gp_unc_masked = np.where(mbes_valid_mask, gp_unc_data, np.nan)

    masked_unc_obj = unc_dbm.copy()
    masked_unc_obj.data = gp_unc_masked

    # raw_data = dbm.data
    raw_data = gp_masked
    if isinstance(raw_data, np.ma.MaskedArray):
        nodata_mask = np.ma.getmaskarray(raw_data)
        elevation = raw_data.filled(np.nan)
    else:
        elevation = np.asarray(raw_data, dtype=float)
        nodata_mask = np.isnan(elevation)

    # raw_unc = unc_dbm.data
    raw_unc = gp_unc_masked
    if isinstance(raw_unc, np.ma.MaskedArray):
        nodata_mask = np.ma.getmaskarray(raw_unc)
        unc = raw_unc.filled(np.nan)
    else:
        unc = np.asarray(raw_unc, dtype=float)
        nodata_mask = np.isnan(unc)

    # Compute cut = auth_depth + elevation (elevation is negative)
    cut = auth_depth + elevation
    cut = np.where(cut > 0, cut, 0.0)
    cut[nodata_mask] = 0.0

    cut_up = cut + (1.96*unc)
    cut_up = np.where(cut_up > 0, cut_up, 0.0)
    cut_up[nodata_mask] = 0.0

    cut_lo = cut - (1.96*unc)
    cut_lo = np.where(cut_lo > 0, cut_lo, 0.0)
    cut_lo[nodata_mask] = 0.0

    px, py = abs(masked_gp_dbm.transform.a), abs(masked_gp_dbm.transform.e)
    volume_ft3 = np.sum(cut) * px * py
    volume_ft3_up = np.sum(cut_up) * px * py
    volume_ft3_lo = np.sum(cut_lo) * px * py
    volume_cy = volume_ft3 / 27.0
    volume_cy_up = volume_ft3_up / 27.0
    volume_cy_lo = volume_ft3_lo / 27.0

    gp_dredge_cuts[path.stem[8:-6]] = (volume_cy, volume_cy * price_per_cy,
                                       volume_cy_up, volume_cy_up * price_per_cy,
                                       volume_cy_lo, volume_cy_lo * price_per_cy,
                                       masked_gp_dbm, masked_unc_obj,
                                       masked_mbes_dbm - dbm)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, ax = plt.subplots(figsize=(4, 0.6), dpi=300)
fig.subplots_adjust(bottom=0.5)

vmin, vmax = -11, 0  # true data min goes a bit past -10
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
cmap = plt.get_cmap('viridis')

cbar = mpl.colorbar.ColorbarBase(
    ax,
    cmap=cmap,
    norm=norm,
    orientation='horizontal'
)

# Force ticks at clean round numbers even though vmin isn't exactly -10
cbar.set_ticks([0, -2, -4, -6, -8, -10])
cbar.set_label('Depth [m]', fontsize=11)
cbar.ax.tick_params(labelsize=9)

plt.show()

In [ ]:
from matplotlib_scalebar.scalebar import ScaleBar

transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1]  # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100]  # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

# --- Compute a global elevation range across all DEMs so the shared colorbar is correct ---
global_vmin = np.inf
global_vmax = -np.inf
for tspace in transect_spacings_to_plot:
    for pspace in point_spacings_to_plot:
        dem_key = f'{tspace * 25}_{pspace}_SVGP'
        global_vmin = min(global_vmin, gp_dredge_cuts[dem_key][6].get_stats('min'))
        global_vmax = max(global_vmax, gp_dredge_cuts[dem_key][6].get_stats('max'))
del dem_key
# --------------------------------------------------------------------------------------

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi=300)
axes = np.atleast_2d(axes)  # ensures axes is always 2D, even if one dimension is 1

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        # --- Grab the pre-built DEM and plot the raster itself, using the shared global scale ---
        dem_key = f'{tspace * 25}_{pspace}_SVGP'
        dem_this_tspace_pspace = gp_dredge_cuts[dem_key][6]

        dem_this_tspace_pspace.plot(
            ax=axes[itspace, ipspace],
            cmap='viridis',
            add_cbar=False,
            vmin=global_vmin,
            vmax=global_vmax,
        )
        # -----------------------------------------------------------------------------------------

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot_local = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot_local)].reset_index(drop=True)

        for transect in transects_to_plot_local:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace, ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace, ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace, ipspace].set_aspect('equal')
        axes[itspace, ipspace].set_xticklabels([])
        axes[itspace, ipspace].set_yticklabels([])
        axes[itspace, ipspace].tick_params(left=False, bottom=False)

        if ipspace == 0:
            axes[itspace, ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace, ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

# --- Layout: run tight_layout first, THEN reserve margin and place sup-labels, so they aren't overridden ---
plt.tight_layout()
fig.subplots_adjust(right=0.83, bottom=0.20, wspace=0.08, hspace=0.08)
fig.supxlabel('Point spacing on transect', fontsize=15, fontweight='bold', y=0.12, x=0.45)
fig.canvas.draw()
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
grid_bottom = min(ax.get_position().y0 for ax in axes.flat)
grid_top = max(ax.get_position().y1 for ax in axes.flat)
fig.text(grid_left - 0.07, (grid_bottom + grid_top) / 2, 'Transect spacing', fontsize=15, fontweight='bold', rotation='vertical', va='center', ha='center')
# ------------------------------------------------------------------------------------------------------

# Use an explicit shared mappable so the colorbar is guaranteed to use the same global limits.
shared_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=global_vmin, vmax=global_vmax), cmap='viridis')
shared_mappable.set_array([])

cbar_ax = fig.add_axes([grid_right + 0.02, grid_bottom, 0.03, grid_top - grid_bottom])
cbar = fig.colorbar(shared_mappable, cax=cbar_ax)
cbar.set_label('Elevation (ft MLLW)', fontsize=13)
cbar.ax.tick_params(rotation=90)

fig.suptitle("Gaussian Process Mean Prediction", fontsize=16, fontweight="bold", y=1.01)

plt.show()

In [ ]:
transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1]  # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100]  # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

# --- Compute a global elevation range across all DEMs so the shared colorbar is correct ---
global_vmin = np.inf
global_vmax = -np.inf
for tspace in transect_spacings_to_plot:
    for pspace in point_spacings_to_plot:
        dem_key = f'{tspace * 25}_{pspace}_SVGP'
        global_vmin = min(global_vmin, gp_dredge_cuts[dem_key][8].get_stats('min'))
        global_vmax = max(global_vmax, gp_dredge_cuts[dem_key][8].get_stats('max'))
del dem_key
# --------------------------------------------------------------------------------------

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi=300)
axes = np.atleast_2d(axes)  # ensures axes is always 2D, even if one dimension is 1

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        # --- Grab the pre-built DEM and plot the raster itself, using the shared global scale ---
        dem_key = f'{tspace * 25}_{pspace}_SVGP'
        dem_this_tspace_pspace = gp_dredge_cuts[dem_key][8]

        dem_this_tspace_pspace.plot(
            ax=axes[itspace, ipspace],
            cmap='PuOr',
            add_cbar=False,
            vmin=global_vmin,
            vmax=global_vmax,
        )
        # -----------------------------------------------------------------------------------------

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot_local = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot_local)].reset_index(drop=True)

        for transect in transects_to_plot_local:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace, ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace, ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace, ipspace].set_aspect('equal')
        axes[itspace, ipspace].set_xticklabels([])
        axes[itspace, ipspace].set_yticklabels([])
        axes[itspace, ipspace].tick_params(left=False, bottom=False)

        if ipspace == 0:
            axes[itspace, ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace, ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

# --- Layout: run tight_layout first, THEN reserve margin and place sup-labels, so they aren't overridden ---
plt.tight_layout()
fig.subplots_adjust(right=0.83, bottom=0.20, wspace=0.08, hspace=0.08)
fig.supxlabel('Point spacing on transect', fontsize=15, fontweight='bold', y=0.12, x=0.45)
fig.canvas.draw()
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
grid_bottom = min(ax.get_position().y0 for ax in axes.flat)
grid_top = max(ax.get_position().y1 for ax in axes.flat)
fig.text(grid_left - 0.07, (grid_bottom + grid_top) / 2, 'Transect spacing', fontsize=15, fontweight='bold', rotation='vertical', va='center', ha='center')
# ------------------------------------------------------------------------------------------------------

# Use an explicit shared mappable so the colorbar is guaranteed to use the same global limits.
shared_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=global_vmin, vmax=global_vmax), cmap='PuOr')
shared_mappable.set_array([])

cbar_ax = fig.add_axes([grid_right + 0.02, grid_bottom, 0.03, grid_top - grid_bottom])
cbar = fig.colorbar(shared_mappable, cax=cbar_ax)
cbar.set_label('Error (ft)', fontsize=13)
cbar.ax.tick_params(rotation=90)

fig.suptitle("Gaussian Process Error", fontsize=16, fontweight="bold", y=1.01)

plt.show()

In [ ]:
transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1]  # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100]  # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

# --- Compute a global elevation range across all DEMs so the shared colorbar is correct ---
global_vmin = np.inf
global_vmax = -np.inf
for tspace in transect_spacings_to_plot:
    for pspace in point_spacings_to_plot:
        dem_key = f'{tspace * 25}_{pspace}_SVGP'
        global_vmin = min(global_vmin, gp_dredge_cuts[dem_key][7].get_stats('min'))
        global_vmax = max(global_vmax, gp_dredge_cuts[dem_key][7].get_stats('max'))
del dem_key
# --------------------------------------------------------------------------------------

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi=300)
axes = np.atleast_2d(axes)  # ensures axes is always 2D, even if one dimension is 1

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        # --- Grab the pre-built DEM and plot the raster itself, using the shared global scale ---
        dem_key = f'{tspace * 25}_{pspace}_SVGP'
        dem_this_tspace_pspace = gp_dredge_cuts[dem_key][7]

        dem_this_tspace_pspace.plot(
            ax=axes[itspace, ipspace],
            cmap='magma_r',
            add_cbar=False,
            vmin=global_vmin,
            vmax=global_vmax,
        )
        # -----------------------------------------------------------------------------------------

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot_local = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot_local)].reset_index(drop=True)

        for transect in transects_to_plot_local:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace, ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace, ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace, ipspace].set_aspect('equal')
        axes[itspace, ipspace].set_xticklabels([])
        axes[itspace, ipspace].set_yticklabels([])
        axes[itspace, ipspace].tick_params(left=False, bottom=False)

        if ipspace == 0:
            axes[itspace, ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace, ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

# --- Layout: run tight_layout first, THEN reserve margin and place sup-labels, so they aren't overridden ---
plt.tight_layout()
fig.subplots_adjust(right=0.83, bottom=0.20, wspace=0.08, hspace=0.08)
fig.supxlabel('Point spacing on transect', fontsize=15, fontweight='bold', y=0.12, x=0.45)
fig.canvas.draw()
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
grid_bottom = min(ax.get_position().y0 for ax in axes.flat)
grid_top = max(ax.get_position().y1 for ax in axes.flat)
fig.text(grid_left - 0.07, (grid_bottom + grid_top) / 2, 'Transect spacing', fontsize=15, fontweight='bold', rotation='vertical', va='center', ha='center')
# ------------------------------------------------------------------------------------------------------

# Use an explicit shared mappable so the colorbar is guaranteed to use the same global limits.
shared_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=global_vmin, vmax=global_vmax), cmap='magma_r')
shared_mappable.set_array([])

cbar_ax = fig.add_axes([grid_right + 0.02, grid_bottom, 0.03, grid_top - grid_bottom])
cbar = fig.colorbar(shared_mappable, cax=cbar_ax)
cbar.set_label('Uncertainty (1σ, ft)', fontsize=13)
cbar.ax.tick_params(rotation=90)
for tick_label in cbar.ax.get_yticklabels():
    x_pos, y_pos = tick_label.get_position()
    tick_label.set_position((x_pos, y_pos + 0.01))
    tick_label.set_va('center')

fig.suptitle("Gaussian Process Uncertainty", fontsize=16, fontweight="bold", y=1.01)

plt.show()

# Generate a TIN vs GP plot to show differences between the default and geostatistical at various transect and sounding spacings

In [ ]:
dredge_cuts

In [ ]:
dredge_cuts[f'SBES_{4* 25}_{20}'][3].get_stats('RMSE')

In [ ]:
ipr_diffs = {}

for t_spacing in transect_spacings_to_plot:
    for p_spacing in point_spacings_to_plot:
        sbes_dbm = dredge_cuts[f'SBES_{t_spacing * 25}_{p_spacing}'][2]
        gp_dbm = gp_dredge_cuts[f'{t_spacing * 25}_{p_spacing}_SVGP'][-3]

        sbes_ref_rmse = dredge_cuts[f'SBES_{t_spacing * 25}_{p_spacing}'][3].get_stats('RMSE')
        gp_ref_rmse = gp_dredge_cuts[f'{t_spacing * 25}_{p_spacing}_SVGP'][-1].get_stats('RMSE')

        ipr_diffs[f'{t_spacing*25}_{p_spacing}'] = [sbes_dbm - gp_dbm, sbes_ref_rmse, gp_ref_rmse]

ipr_diffs

In [ ]:
transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1]  # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100]  # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

# --- Compute a global elevation range across all DEMs so the shared colorbar is correct ---
global_vmin = np.inf
global_vmax = -np.inf
for tspace in transect_spacings_to_plot:
    for pspace in point_spacings_to_plot:
        dem_key = f'{tspace * 25}_{pspace}'
        global_vmin = min(global_vmin, ipr_diffs[dem_key][0].get_stats('min'))
        global_vmax = max(global_vmax, ipr_diffs[dem_key][0].get_stats('max'))
del dem_key

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi=300)
axes = np.atleast_2d(axes)  # ensures axes is always 2D, even if one dimension is 1

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        # --- Grab the pre-built DEM and plot the raster itself, using the shared global scale ---
        dem_key = f'{tspace * 25}_{pspace}'
        dem_this_tspace_pspace = ipr_diffs[dem_key][0]

        dem_this_tspace_pspace.plot(
            ax=axes[itspace, ipspace],
            cmap='PuOr',
            add_cbar=False,
            vmin=global_vmin,
            vmax=global_vmax,
        )
        

        tin_rmse = ipr_diffs[dem_key][1]
        gp_rmse = ipr_diffs[dem_key][2]

        rmse_text = f'TIN RMSE = {tin_rmse:.2f} ft\nGP RMSE = {gp_rmse:.2f} ft'
        axes[itspace, ipspace].text(
            0.02, 0.03, rmse_text,
            transform=axes[itspace, ipspace].transAxes,
            fontsize=6.5,
            va='bottom', ha='left',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.75, edgecolor='none'),
            zorder=5
        )
        

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot_local = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot_local)].reset_index(drop=True)

        for transect in transects_to_plot_local:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace, ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace, ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace, ipspace].set_aspect('equal')
        axes[itspace, ipspace].set_xticklabels([])
        axes[itspace, ipspace].set_yticklabels([])
        axes[itspace, ipspace].tick_params(left=False, bottom=False)

        if ipspace == 0:
            axes[itspace, ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace, ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

# --- Layout: run tight_layout first, THEN reserve margin and place sup-labels, so they aren't overridden ---
plt.tight_layout()
fig.subplots_adjust(right=0.83, bottom=0.20, wspace=0.08, hspace=0.08)
fig.supxlabel('Point spacing on transect', fontsize=15, fontweight='bold', y=0.12, x=0.45)
fig.canvas.draw()
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
grid_bottom = min(ax.get_position().y0 for ax in axes.flat)
grid_top = max(ax.get_position().y1 for ax in axes.flat)
fig.text(grid_left - 0.07, (grid_bottom + grid_top) / 2, 'Transect spacing', fontsize=15, fontweight='bold', rotation='vertical', va='center', ha='center')

# Use an explicit shared mappable so the colorbar is guaranteed to use the same global limits.
shared_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=global_vmin, vmax=global_vmax), cmap='PuOr')
shared_mappable.set_array([])

cbar_ax = fig.add_axes([grid_right + 0.02, grid_bottom, 0.03, grid_top - grid_bottom])
cbar = fig.colorbar(shared_mappable, cax=cbar_ax)
cbar.set_label('Difference (TIN-GP) (ft, MLLW)', fontsize=13)
cbar.ax.tick_params(rotation=90)

fig.suptitle("TIN - Gaussian Process Difference", fontsize=16, fontweight="bold", y=1.01)

plt.show()

In [ ]:
transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1]  # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100]  # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

# --- Compute a global elevation range across all DEMs so the shared colorbar is correct ---
global_vmin = np.inf
global_vmax = -np.inf
for tspace in transect_spacings_to_plot:
    for pspace in point_spacings_to_plot:
        dem_key = f'{tspace * 25}_{pspace}'
        global_vmin = min(global_vmin, ipr_diffs[dem_key][0].get_stats('min'))
        global_vmax = max(global_vmax, ipr_diffs[dem_key][0].get_stats('max'))
del dem_key
# --------------------------------------------------------------------------------------

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi=300)
axes = np.atleast_2d(axes)  # ensures axes is always 2D, even if one dimension is 1

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        # --- Grab the pre-built DEM and plot the raster itself, using the shared global scale ---
        dem_key = f'{tspace * 25}_{pspace}'
        dem_this_tspace_pspace = ipr_diffs[dem_key][0]

        dem_this_tspace_pspace.plot(
            ax=axes[itspace, ipspace],
            cmap='PuOr',
            add_cbar=False,
            vmin=global_vmin,
            vmax=global_vmax,
        )
        # -----------------------------------------------------------------------------------------

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot_local = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot_local)].reset_index(drop=True)

        for transect in transects_to_plot_local:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace, ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace, ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace, ipspace].set_aspect('equal')
        axes[itspace, ipspace].set_xticklabels([])
        axes[itspace, ipspace].set_yticklabels([])
        axes[itspace, ipspace].tick_params(left=False, bottom=False)

        if ipspace == 0:
            axes[itspace, ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace, ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

# --- Layout: run tight_layout first, THEN reserve margin and place sup-labels, so they aren't overridden ---
plt.tight_layout()
fig.subplots_adjust(right=0.83, bottom=0.20, wspace=0.08, hspace=0.08)
fig.supxlabel('Point spacing on transect', fontsize=15, fontweight='bold', y=0.12, x=0.45)
fig.canvas.draw()
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
grid_bottom = min(ax.get_position().y0 for ax in axes.flat)
grid_top = max(ax.get_position().y1 for ax in axes.flat)
fig.text(grid_left - 0.07, (grid_bottom + grid_top) / 2, 'Transect spacing', fontsize=15, fontweight='bold', rotation='vertical', va='center', ha='center')
# ------------------------------------------------------------------------------------------------------

# Use an explicit shared mappable so the colorbar is guaranteed to use the same global limits.
shared_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=global_vmin, vmax=global_vmax), cmap='PuOr')
shared_mappable.set_array([])

cbar_ax = fig.add_axes([grid_right + 0.02, grid_bottom, 0.03, grid_top - grid_bottom])
cbar = fig.colorbar(shared_mappable, cax=cbar_ax)
cbar.set_label('Difference (TIN-GP) (ft, MLLW)', fontsize=13)
cbar.ax.tick_params(rotation=90)

fig.suptitle("TIN - Gaussian Process Difference", fontsize=16, fontweight="bold", y=1.01)

plt.show()

In [ ]:
gp_bathy_paths

In [ ]:
clipped_sbes_dbm_paths

In [ ]:
for path in clipped_sbes_dbm_paths:
    dbm = xdem.DEM(path).reproject(masked_mbes_dbm)

    elevation = dbm.data

    # Explicitly determine which cells have real data
    if np.ma.is_masked(elevation):
        valid = ~np.ma.getmaskarray(elevation)
    else:
        valid = np.isfinite(elevation)

    elevation_filled = np.ma.filled(elevation, 0.0) if np.ma.is_masked(elevation) else elevation

    # Compute cut = auth_depth + elevation (elevation is negative)
    cut = auth_depth + elevation_filled
    cut = np.where(cut > 0, cut, 0.0)
    cut[~valid] = 0.0    # <-- zero out anything without real survey data

    px, py = abs(dbm.transform.a), abs(dbm.transform.e)
    volume_ft3 = np.sum(cut) * px * py
    volume_cy = volume_ft3 / 27.0

    dredge_cuts[path.stem[8:-10]] = (volume_cy, volume_cy * price_per_cy, dbm, masked_mbes_dbm - dbm) #masked_mbes_dbm - dbm

In [ ]:
# NOTE: [0] = mean vol, [1] = mean price, [2]=upper vol, [3] = upper price, [4]=lower vol, [5]=lower price
gp_diffs = []
for k, (mean_vol, mean_price, up_vol, up_price, lo_vol, lo_price, _, _, _) in gp_dredge_cuts.items():
    ts, ss, _ = k.split("_")
    # vol_pct_diff = (np.abs(base_cut - vol) / ((base_cut + vol)/2))*100
    # price_pct_diff = (np.abs(base_cut - price) / ((base_cut + price)/2))*100
    mean_vol_diff = base_cut - mean_vol
    mean_price_diff = base_price - mean_price

    up_vol_diff = base_cut - up_vol
    up_price_diff = base_price - up_price

    lo_vol_diff = base_cut - lo_vol
    lo_price_diff = base_price - lo_price

    gp_diffs.append({
        'Track_Spacing': ts,
        'Sounding_Spacing': ss,
        'Mean_Vol_Diff': mean_vol_diff,
        'Mean_Cost_Diff': mean_price_diff,
        'Up_Vol_Diff': up_vol_diff,
        'Up_Cost_Diff': up_price_diff,
        'Lo_Vol_Diff': lo_vol_diff,
        'Lo_Cost_Diff': lo_price_diff
    })

gp_df = pd.DataFrame(gp_diffs)

In [ ]:
gp_df

In [ ]:
sampled_df = df[df['Track_Spacing'].isin(['300', '100', '25'])]
sampled_df = sampled_df[sampled_df['Sounding_Spacing'].isin(['4', '20', '100'])]
sampled_df

In [ ]:
merged_sampled_df = pd.merge(sampled_df, gp_df, on=['Track_Spacing', 'Sounding_Spacing'])
merged_sampled_df

In [ ]:
merged_sampled_df['Track_Spacing'] = merged_sampled_df['Track_Spacing'].astype(float)
merged_sampled_df['Sounding_Spacing'] = merged_sampled_df['Sounding_Spacing'].astype(float)

# Absolute-value versions of everything, so bars are all comparable
merged_sampled_df['Vol_Diff_Abs'] = merged_sampled_df['Vol_Diff'].abs()
merged_sampled_df['Cost_Diff_Abs'] = merged_sampled_df['Cost_Diff'].abs()

merged_sampled_df['Mean_Vol_Diff_Abs'] = merged_sampled_df['Mean_Vol_Diff'].abs()
merged_sampled_df['Mean_Cost_Diff_Abs'] = merged_sampled_df['Mean_Cost_Diff'].abs()

track_spacing = sorted(merged_sampled_df['Track_Spacing'].unique())
sounding_spacing = sorted(merged_sampled_df['Sounding_Spacing'].unique())

fig, axes = plt.subplots(2, 1, figsize=(14, 9), dpi=300)
colors = plt.cm.viridis(np.linspace(0, 1, len(sounding_spacing)))
n = len(sounding_spacing)
# each sounding-spacing group now has 2 sub-bars (single-run + mean)
sub_width = 0.8 / (n * 2)
x = np.arange(len(track_spacing))

baseline_labels = {
    'Vol_Diff': f'Baseline Volume: {base_cut:,.0f} CY',
    'Cost_Diff': f'Baseline Cost: ${base_price:,.0f}',
}

plot_spec = [
    ('Vol_Diff_Abs', 'Mean_Vol_Diff_Abs',
     'Dredge Cut Volume Absolute Difference', '|Volume Diff| (CY)', 'Vol_Diff'),
    ('Cost_Diff_Abs', 'Mean_Cost_Diff_Abs',
     'Estimated Cost Absolute Difference (at $25/CY)', '|Cost Diff| ($)', 'Cost_Diff'),
]

for ax, (single_col, mean_col, title, ylabel, baseline_key) in zip(axes, plot_spec):
    for i, (ss, c) in enumerate(zip(sounding_spacing, colors)):
        sub = merged_sampled_df[merged_sampled_df['Sounding_Spacing'] == ss].set_index('Track_Spacing')
        sub = sub.reindex(track_spacing)

        single_vals = sub[single_col].values
        mean_vals = sub[mean_col].values

        # center offset for this sounding-spacing's pair of bars
        group_center = (i - n / 2) * (2 * sub_width) + sub_width

        # bar 1: original single-run diff (solid)
        ax.bar(x + group_center - sub_width / 2, single_vals, width=sub_width, color=c,
               edgecolor='white', linewidth=0.3,
               label=f'{int(ss)} ft (TIN)' if ax is axes[0] else None)

        # bar 2: mean diff (hatched, same color)
        ax.bar(x + group_center + sub_width / 2, mean_vals, width=sub_width, color=c,
               edgecolor='black', linewidth=0.5, hatch='//', alpha=0.75,
               label=f'{int(ss)} ft (GP)' if ax is axes[0] else None)

    ax.axhline(0, color='black', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{int(t)}' for t in track_spacing], fontsize=13)
    ax.tick_params(axis='y', labelsize=13)
    ax.set_xlabel('Transect spacing (ft)', fontweight='bold', fontsize=15)
    ax.set_ylabel(ylabel, fontweight='bold', fontsize=15)
    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')

    ax.text(
        0.02, 0.96, baseline_labels[baseline_key],
        transform=ax.transAxes,
        fontsize=13, fontweight='bold', va='top', ha='left',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor='black', alpha=0.85)
    )

leg = axes[0].legend(
    title='Point spacing',
    bbox_to_anchor=(1.01, 1),
    loc='upper left',
    prop={'size': 10, 'weight': 'bold'},
    title_fontsize=11,
    ncol=1
)
leg.get_title().set_weight('bold')
plt.tight_layout()
# plt.savefig('/mnt/user-data/outputs/bar_chart.png', dpi=300, bbox_inches='tight')
plt.show()